In [ ]:
!git clone https://github.com/bindhupagadala10/it-rca-capa-framework.git
!pip install -q unsloth
!pip install -q datasets transformers accelerate trl peft
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

In [ ]:
from unsloth import FastLanguageModel
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = 4096,
    load_in_4bit = True,
)

In [ ]:
max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
train_dataset = load_dataset(
    "json",
    data_files="/content/it-rca-capa-framework/data/splits/rca_train.jsonl",
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files="/content/it-rca-capa-framework/data/splits/rca_val.jsonl",
    split="train"
)

print(train_dataset)
print(val_dataset)

In [ ]:
def formatting_prompts_func(examples):

    texts = []

    for msgs in examples["messages"]:

        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)

    return {
        "text": texts,
    }

train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
)

val_dataset = val_dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        learning_rate=2e-4,
        num_train_epochs=3,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir="qwen_rca_output",
        report_to="none",
    ),
)

In [ ]:
print(trainer.state.global_step)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("qwen_rca_adapter")
tokenizer.save_pretrained("qwen_rca_adapter")

In [ ]:
!zip -r qwen_rca_adapter.zip qwen_rca_adapter

In [ ]:
!ls -lh qwen_rca_adapter.zip


In [ ]:
import json

with open("/content/it-rca-capa-framework/data/splits/rca_test.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

input_text = sample["messages"][0]["content"]
ground_truth = sample["messages"][1]["content"]

print("=== INPUT ===\n")
print(input_text[:3000])

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": input_text
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    inputs,
    max_new_tokens=1200,
    temperature=0.2,
    do_sample=False,
)

generated = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(generated)

In [ ]:
print("\n\n=== GROUND TRUTH ===\n")
print(ground_truth)

In [ ]:
import json
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

results = []

with open(
    "/content/it-rca-capa-framework/data/splits/rca_test.jsonl",
    "r",
    encoding="utf-8"
) as f:

    test_data = [json.loads(line) for line in f]

for i in range(10):

    sample = test_data[i]

    user_input = sample["messages"][0]["content"]
    ground_truth = sample["messages"][1]["content"]

    messages = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=1200,
        temperature=0.2,
        do_sample=False
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    results.append({
        "id": i + 1,
        "prediction": prediction,
        "ground_truth": ground_truth
    })

with open("rca_predictions_10.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved rca_predictions_10.json")

In [ ]:
from google.colab import files
files.download("rca_predictions_10.json")

In [ ]:
!pip install -q rouge-score bert-score pandas matplotlib

In [ ]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ONLY EVALUATE 20 TEST RECORDS
N_SAMPLES = 20

test_file = "/content/it-rca-capa-framework/data/splits/rca_test.jsonl"

with open(test_file, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f][:N_SAMPLES]

predictions = []
references = []

for sample in tqdm(test_data):

    prompt = sample["messages"][0]["content"]
    ground_truth = sample["messages"][1]["content"]

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        inputs,
        max_new_tokens=800,
        temperature=0.2,
        do_sample=False,
        use_cache=True,
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    predictions.append(prediction)
    references.append(ground_truth)

print(f"Generated {len(predictions)} predictions.")

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(predictions, references):

    scores = scorer.score(ref, pred)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

rouge1 = sum(rouge1_scores) / len(rouge1_scores)
rouge2 = sum(rouge2_scores) / len(rouge2_scores)
rougeL = sum(rougeL_scores) / len(rougeL_scores)

print("ROUGE-1:", round(rouge1, 4))
print("ROUGE-2:", round(rouge2, 4))
print("ROUGE-L:", round(rougeL, 4))

In [ ]:
from bert_score import score as bertscore

P, R, F1 = bertscore(
    predictions,
    references,
    lang="en",
    verbose=True
)

bert_precision = P.mean().item()
bert_recall = R.mean().item()
bert_f1 = F1.mean().item()

print("BERTScore Precision:", round(bert_precision, 4))
print("BERTScore Recall   :", round(bert_recall, 4))
print("BERTScore F1       :", round(bert_f1, 4))

In [ ]:
import pandas as pd

metrics = {
    "ROUGE-1": round(rouge1, 4),
    "ROUGE-2": round(rouge2, 4),
    "ROUGE-L": round(rougeL, 4),
    "BERTScore_P": round(bert_precision, 4),
    "BERTScore_R": round(bert_recall, 4),
    "BERTScore_F1": round(bert_f1, 4),
}

df = pd.DataFrame([metrics])

df.to_csv(
    "rca_metrics.csv",
    index=False
)

print(df)

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "BERTScore"
]

values = [
    metrics["ROUGE-1"],
    metrics["ROUGE-2"],
    metrics["ROUGE-L"],
    metrics["BERTScore_F1"]
]

plt.figure(figsize=(8,5))

plt.bar(labels, values)

plt.title("Qwen RCA Evaluation")
plt.ylabel("Score")
plt.ylim(0,1)

for i, v in enumerate(values):
    plt.text(i, v + 0.02, str(round(v,3)), ha="center")

plt.show()

In [ ]:
from google.colab import files

files.download("rca_metrics.csv")

In [ ]:
plt.savefig("qwen_rca_evaluation.png", dpi=300, bbox_inches="tight")

In [ ]:
from google.colab import files
files.download("qwen_rca_evaluation.png")
